In [21]:
import os

# This prints the exact folder your notebook is running from
print(os.getcwd())

# This lists all files the notebook can actually see in that folder
print(os.listdir("."))

/home/Conner/Projects/stuff_plus
['.git', 'LICENSE', '.venv', 'requirements.txt', 'todo.md', 'notebooks', "csv's", '.vscode', 'models', '.gitignore', 'premier_model', 'statcast_pull.py', 'README.md', 'note.txt']


## Import CSV's and Python Libraries

Run `pip install -r requirements.txt` in your terminal to install the packages necessary to run this notebook on your local machine


In [22]:
import pandas as pd
import numpy as np
import xgboost as xgb
from xgboost import XGBRegressor
import os

# Load one pre-trained XGBoost model per pitch type from the /models directory.
# Each .ubj file was trained on MLB StatCast 2024-2026 data using delta run value
# as the target variable, with the 8 features below as inputs.
MODELS = {}
MODEL_PITCH_TYPES = ['CB', 'CH', 'CT', 'FB', 'KN', 'SI', 'SL', 'SP', 'SW']
MODEL_DIR = 'models'

import os

for pt in MODEL_PITCH_TYPES:
    file_path = os.path.join(MODEL_DIR, f"{pt}.ubj")
    
    # Check if the file actually exists
    if not os.path.exists(file_path):
        print(f"Error: Model file missing for pitch type '{pt}' at path: {file_path}")
        continue # Skip this pitch type, or use 'break' to stop
        
    m = XGBRegressor()
    m.load_model(file_path)
    MODELS[pt] = m

print(f"Loaded {len(MODELS)} models: {list(MODELS.keys())}")

df = pd.read_csv("csv's/tm_stuff_plus.csv")
print(f"\nLoaded {len(df):,} rows × {df.shape[1]} columns")
df.head()

Loaded 9 models: ['CB', 'CH', 'CT', 'FB', 'KN', 'SI', 'SL', 'SP', 'SW']

Loaded 20,291 rows × 21 columns


,pitcher_name,pitcher_level,pitcher_throws,date,pitch_type,pitch_number,release_speed,spin_rate,induced_vertical_break,horizontal_break,...,horizontal_approach_angle,vertical_break,spin_efficiency,spin_axis,tilt,extension,release_height,release_side,plate_location_height,plate_location_side
0,Wyatt Gibson,College,Left,2023-11-24,FB,7,78.56706,1797.9653,11.55309,-11.18099,...,0.239720,-34.85415,0.728916,136.02661,10:30:00,4.78048,5.07959,-1.27562,1.02636,-0.02276
1,Wyatt Gibson,College,Left,2023-11-24,FB,8,78.57823,1930.6069,13.41417,-5.73724,...,-0.279315,-33.47556,0.738446,156.90613,11:15:00,4.44279,5.11428,-1.20311,2.62001,-0.93968
2,Wyatt Gibson,College,Left,2023-11-24,FB,9,76.36920,1792.9038,17.66483,-3.36008,...,-0.507281,-32.57337,0.623952,169.24277,11:45:00,4.34863,5.14368,-1.50641,5.08989,-1.67569
3,Wyatt Gibson,College,Left,2023-11-24,SI,10,77.15758,1813.2701,14.13264,-13.68481,...,-1.059715,-36.14442,0.600773,136.01645,10:30:00,3.85140,5.07517,-1.04206,2.05757,-0.78176
4,Wyatt Gibson,College,Left,2023-11-24,SI,11,77.35618,1718.2062,NaN,NaN,...,NaN,NaN,NaN,97.03653,09:15:00,4.64714,4.95282,-1.26937,NaN,NaN


## Clean Data

This cell cleans the data, accounting for missing rows and standardizing metrics by inverting all left-handed pitcher metrics to be right-handed for consistency. It also normalizes the `pitcher_level` field so that team-specific college and pro entries are bucketed into three clean tiers — `High School`, `College`, and `Pro` — which are used later for within-level Stuff+ scaling.

### The 8 features every XGBoost model expects (must match training order)

In [ ]:
FEATURES = [
    'release_speed',
    'spin_rate',
    'induced_vertical_break',
    'horizontal_break',
    'extension',
    'release_height',
    'release_side',
    'spin_axis',
    # Differential features below measure difference from pitcher's avg fb
    'velo_diff',
    'ivb_diff',
    'hb_diff',
    'spin_diff'
]

# Columns to sign-flip for LHP so all pitchers are treated as RHP
FLIP_COLS = ['horizontal_break', 'release_side']

REQUIRED_COLS = ['pitch_type', 'pitcher_throws', 'release_speed',
                 'spin_rate', 'induced_vertical_break', 'horizontal_break']

# Bucket granular level labels into three clean tiers for within-level comparisons
COLLEGE_KEYWORDS = [
    'college', 'university', 'state', 'stix', 'vcu', 'fiu', 'mizzou',
    'ucm', 'siu', 'radford', 'lipscomb', 'bradley', 'gaston', 'brower', 'iowa',
    'memphis', 'jacksonville', 'nc state', 'west virg'
]

def normalize_level(level: str) -> str:
    if pd.isna(level):
        return 'Unknown'
    lower = level.strip().lower()
    if lower == 'high school':
        return 'High School'
    if lower == 'pro':
        return 'Pro'
    if any(kw in lower for kw in COLLEGE_KEYWORDS):
        return 'College'
    # Treat anything with a team/org name that isn't HS or Pro as College
    return 'College'

# Drop warmup pitches
if 'is_warmup' in df.columns:
    df = df[df['is_warmup'] != 1]

# Normalize level labels into three tiers
df['pitcher_level'] = df['pitcher_level'].apply(normalize_level)

# Drop rows missing essential pitch data
df = df.dropna(subset=REQUIRED_COLS).copy()

# Exclude pitchers with undefined handedness — can't normalize direction reliably
df = df[df['pitcher_throws'].isin(['Right', 'Left'])].copy()

# Sign-flip horizontal metrics for LHP so all movement reads as RHP convention
lhp = df['pitcher_throws'] == 'Left'
df.loc[lhp, FLIP_COLS] = df.loc[lhp, FLIP_COLS] * -1

# Flip spin axis for LHP (mirror across 180°)
df.loc[lhp, 'spin_axis'] = (360 - df.loc[lhp, 'spin_axis']) % 360

print(f"Rows after cleaning: {len(df):,}")
print(f"\nLevel distribution after normalization:\n{df['pitcher_level'].value_counts().to_string()}")
print(f"\nPitch type distribution:\n{df['pitch_type'].value_counts().to_string()}")

Rows after cleaning: 17,958

Level distribution after normalization:
pitcher_level
College        12197
High School     5257
Pro              504

Pitch type distribution:
pitch_type
FB    7621
CH    2908
SI    2457
SL    2429
CB    1628
CT     560
SP     274
SW      79
KN       2


## Compute Differential Variables

In [ ]:
# For each pitcher, compute their mean fastball profile, then calculate
# the difference between each non-FB pitch and that fastball average.
# Pitchers with no fastball get NaN diffs (model will handle via median fill).

FB_TYPES = {'FB', 'SI'}  # pitch types treated as "fastball" for diff baseline

fb_avg = (
    df[df['pitch_type'].isin(FB_TYPES)]
    .groupby('pitcher_name')[['release_speed', 'induced_vertical_break',
                               'horizontal_break', 'spin_rate']]
    .mean()
    .rename(columns={
        'release_speed':          'fb_velo',
        'induced_vertical_break': 'fb_ivb',
        'horizontal_break':       'fb_hb',
        'spin_rate':              'fb_spin',
    })
)

df = df.join(fb_avg, on='pitcher_name')

# Diffs: fastball value minus offspeed value
# Positive = FB is faster/has more of that attribute than the offspeed pitch
df['velo_diff'] = df['fb_velo'] - df['release_speed']
df['ivb_diff']  = df['fb_ivb']  - df['induced_vertical_break']
df['hb_diff']   = df['fb_hb']   - df['horizontal_break']
df['spin_diff'] = df['fb_spin'] - df['spin_rate']

# Zero out diffs for fastballs themselves — no meaningful differential
fb_mask = df['pitch_type'].isin(FB_TYPES)
df.loc[fb_mask, ['velo_diff', 'ivb_diff', 'hb_diff', 'spin_diff']] = 0

# Drop the temporary fb_* baseline columns
df.drop(columns=['fb_velo', 'fb_ivb', 'fb_hb', 'fb_spin'], inplace=True)

print("Differential features added.")
print(df[['pitch_type', 'velo_diff', 'ivb_diff', 'hb_diff', 'spin_diff']]
      .groupby('pitch_type').mean().round(2).to_string())

## Run XGBoost Predictions

Each pitch gets a raw predicted run-value score from the pre-trained XGBoost model that matches its pitch type. The model was trained on MLB StatCast data (2024–2026) using **delta run value (ΔRV)** as the target variable, so the raw output represents the expected change in run value associated with that pitch's physical characteristics — lower (more negative) predicted ΔRV means a more effective pitch.

The 8 input features used by every model are:
- `release_speed` — velocity out of the hand
- `spin_rate` — raw spin in RPM
- `induced_vertical_break` — vertical movement generated by spin (inches)
- `horizontal_break` — horizontal movement (inches, RHP convention after flip)
- `extension` — release distance toward home plate (feet)
- `release_height` — vertical release point (feet)
- `release_side` — horizontal release point (feet, RHP convention after flip)
- `spin_axis` — clock-face spin direction (degrees, RHP convention after flip)

Pitches whose pitch type does not have a corresponding model are skipped and will have a null `predicted_rv`.

In [24]:
# Predict raw ΔRV for every pitch using the matching pitch-type model
df['predicted_rv'] = np.nan

for pt, model in MODELS.items():
    mask = df['pitch_type'] == pt
    if mask.sum() == 0:
        continue

    X = df.loc[mask, FEATURES].copy()

    # Fill any remaining nulls in optional features with the column median
    # so the model always receives a complete 8-feature row
    for col in FEATURES:
        if X[col].isnull().any():
            X[col] = X[col].fillna(X[col].median())

    df.loc[mask, 'predicted_rv'] = model.predict(X)

covered = df['predicted_rv'].notna().sum()
skipped = df['predicted_rv'].isna().sum()
print(f"Predictions generated: {covered:,} pitches")
print(f"Skipped (no model for pitch type): {skipped:,} pitches")
print(f"\nPredicted ΔRV stats:")
print(df['predicted_rv'].describe().round(5).to_string())


Predictions generated: 17,958 pitches
Skipped (no model for pitch type): 0 pitches

Predicted ΔRV stats:
count    17958.00000
mean         0.49487
std          0.29968
min         -0.15695
25%          0.28710
50%          0.49059
75%          0.64432
max          2.51800


## Scale to Stuff+

Raw predicted ΔRV scores are converted to the standard **Stuff+ scale** where **100 = average** and **each 10 points = 10% better/worse than average** (one standard deviation).

Because the raw score is a run value — lower (more negative) is better — the sign is **flipped** before scaling so that a higher Stuff+ number always means better stuff.

Pitchers are **only compared against others at their own level** (High School vs. High School, College vs. College, Pro vs. Pro). This prevents a high school pitcher from being penalized against pro-caliber velocity, and ensures a pro arm isn't graded on a curve against HS averages. The population mean and standard deviation used for scaling are derived from the **pitch-type group within that level** — so a HS curveball is benchmarked against all HS curveballs, a college fastball against all college fastballs, and so on.

In [25]:
df['stuff_plus'] = np.nan

# Scale within each (level × pitch_type) group independently
for (level, pt), grp in df.groupby(['pitcher_level', 'pitch_type']):
    raw = df.loc[grp.index, 'predicted_rv'].dropna()
    if len(raw) < 2 or raw.std() == 0:
        continue

    pop_mean = raw.mean()
    pop_std  = raw.std()

    # Flip sign: more negative predicted_rv → better pitch → higher Stuff+
    scaled = 100 + ((pop_mean - raw) / pop_std) * 10
    df.loc[scaled.index, 'stuff_plus'] = scaled

print("Stuff+ distribution by level and pitch type:")
print(
    df.dropna(subset=['stuff_plus'])
    .groupby(['pitcher_level', 'pitch_type'])['stuff_plus']
    .describe()[['count', 'mean', 'std', 'min', '50%', 'max']]
    .round(1)
    .to_string()
)

Stuff+ distribution by level and pitch type:
                           count   mean   std   min    50%    max
pitcher_level pitch_type                                         
College       CB           982.0  100.0  10.0  68.0  101.5  127.6
              CH          2028.0  100.0  10.0  49.7   99.2  116.8
              CT           440.0  100.0  10.0  64.1  101.3  117.3
              FB          5049.0  100.0  10.0  66.2   99.0  126.4
              SI          1660.0  100.0  10.0  56.0  102.6  117.9
              SL          1782.0  100.0  10.0  52.3  101.9  119.5
              SP           199.0  100.0  10.0  53.3  100.5  116.2
              SW            56.0  100.0  10.0  73.6  101.9  112.4
High School   CB           617.0  100.0  10.0  52.0   99.3  122.3
              CH           828.0  100.0  10.0  55.9  100.0  118.3
              CT            93.0  100.0  10.0  73.4  102.1  120.0
              FB          2377.0  100.0  10.0  73.3  100.9  134.2
              SI           701.

## Per-Pitcher Summary

Aggregate to one row per pitcher × pitch type, showing average Stuff+ alongside key metrics. This is the primary output table.


In [26]:
summary = (
    df.dropna(subset=['stuff_plus'])
    .groupby(['pitcher_name', 'pitcher_level', 'pitcher_throws', 'pitch_type'], as_index=False)
    .agg(
        pitches    = ('stuff_plus',             'count'),
        stuff_plus = ('stuff_plus',             'mean'),
        velocity   = ('release_speed',          'mean'),
        spin_rate  = ('spin_rate',              'mean'),
        ivb        = ('induced_vertical_break', 'mean'),
        h_break    = ('horizontal_break',       'mean'),
        extension  = ('extension',              'mean'),
    )
    .round(1)
    .sort_values(['pitcher_name', 'stuff_plus'], ascending=[True, False])
    .reset_index(drop=True)
)


pd.set_option('display.max_rows', 200)
pd.set_option('display.width', 140)
display(summary)

,pitcher_name,pitcher_level,pitcher_throws,pitch_type,pitches,stuff_plus,velocity,spin_rate,ivb,h_break,extension
0,Aaron Alston,High School,Right,CB,1,115.4,74.9,2310.0,3.7,-4.5,5.0
1,Aaron Alston,High School,Right,FB,4,109.9,83.3,1993.1,18.8,7.6,5.7
2,Aaron Alston,High School,Right,CH,3,107.1,76.8,1538.4,17.1,14.1,5.6
3,Aaron Alston,High School,Right,SL,2,103.8,76.1,1903.9,6.1,1.5,5.2
4,Aaron Alston,High School,Right,SI,4,101.4,81.1,1810.2,19.3,9.8,5.7
...,...,...,...,...,...,...,...,...,...,...,...
3678,Zander Street,High School,Right,CB,2,112.9,69.5,2148.2,1.6,-10.7,4.4
3679,Zander Street,High School,Right,SP,1,108.7,71.8,878.8,-0.5,5.7,4.0
3680,Zander Street,High School,Right,FB,9,103.9,83.3,2091.1,14.5,11.0,5.0
3681,Zander Street,High School,Right,CH,2,100.7,75.2,1843.4,8.5,19.5,5.0


## Pitcher Lookup

Change `pitcher_name` below to any name exactly as it appears in the data to see every pitch with its Stuff+ score and underlying metrics.

In [27]:
pitcher_name = "Wyatt Gibson"  # ← change to the pitcher you want to look up

LOOKUP_COLS = [
    'date', 'pitch_type', 'release_speed', 'spin_rate',
    'induced_vertical_break', 'horizontal_break',
    'vertical_approach_angle', 'extension', 'predicted_rv', 'stuff_plus',
]

cols_present = [c for c in LOOKUP_COLS if c in df.columns]
pitcher_df   = df.loc[df['pitcher_name'] == pitcher_name, cols_present].copy()

if pitcher_df.empty:
    print(f"No pitches found for '{pitcher_name}'.")
    print("Available pitchers:", sorted(df['pitcher_name'].unique().tolist()))
else:
    pitcher_df = pitcher_df.sort_values(['date', 'pitch_type']).reset_index(drop=True).round(2)
    print(f"{pitcher_name}  —  {len(pitcher_df)} pitches\n")
    display(pitcher_df)

Wyatt Gibson  —  13 pitches



,date,pitch_type,release_speed,spin_rate,induced_vertical_break,horizontal_break,vertical_approach_angle,extension,predicted_rv,stuff_plus
0,2023-11-24,CB,68.99,1875.70,-7.84,-16.38,-10.90,4.10,0.48,103.99
1,2023-11-24,CB,67.55,1926.70,-8.76,-16.95,-8.47,3.86,0.77,92.30
2,2023-11-24,CH,75.75,1714.54,1.16,18.78,-7.54,4.59,0.24,96.20
3,2023-11-24,CH,76.07,1609.10,11.61,14.86,-5.88,4.38,0.27,93.94
4,2023-11-24,CT,73.07,1969.36,6.15,-6.70,-8.48,4.31,0.40,98.38
5,2023-11-24,CT,75.80,1963.59,9.23,-2.55,-8.94,4.35,0.52,94.40
6,2023-11-24,FB,78.57,1797.97,11.55,11.18,-7.68,4.78,0.81,83.83
7,2023-11-24,FB,78.58,1930.61,13.41,5.74,-5.88,4.44,0.66,91.04
8,2023-11-24,FB,76.37,1792.90,17.66,3.36,-3.24,4.35,0.95,76.81
9,2023-11-24,FB,79.42,1909.98,18.80,1.27,-4.67,4.59,0.63,92.39


## Output Data

Saves two CSV files to the `csv's/` folder:
- **`stuff_plus_pitch_level.csv`** — one row per pitch with Stuff+ and core metrics
- **`stuff_plus_summary.csv`** — one row per pitcher × pitch type with aggregated averages

In [28]:
PITCH_LEVEL_COLS = [
    'pitcher_name', 'pitcher_level', 'pitcher_throws', 'date', 'pitch_type',
    'release_speed', 'spin_rate', 'induced_vertical_break', 'horizontal_break',
    'vertical_approach_angle', 'spin_efficiency', 'extension',
    'release_height', 'release_side', 'predicted_rv', 'stuff_plus',
]

pitch_level_cols_present = [c for c in PITCH_LEVEL_COLS if c in df.columns]
out_pitch = df[pitch_level_cols_present].copy()

out_pitch.to_csv("csv's/stuff_plus_pitch_level.csv", index=False)
summary.to_csv("csv's/stuff_plus_summary.csv",      index=False)

print(f"Saved {len(out_pitch):,} pitch-level rows  → csv's/stuff_plus_pitch_level.csv")
print(f"Saved {len(summary):,} pitcher-summary rows → csv's/stuff_plus_summary.csv")

Saved 17,958 pitch-level rows  → csv's/stuff_plus_pitch_level.csv
Saved 3,683 pitcher-summary rows → csv's/stuff_plus_summary.csv
